# Champions League Predictor — Part 2: The Model

Run Part 1 first.

Trains XGBoost, tests it on the whole 2025/26 Champions League season (189 games).

In [15]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.inspection import permutation_importance

from xgboost import XGBClassifier

## Load

Already scaled and encoded in Part 1.

In [16]:
data = np.load('ucl_predictor_data.npz', allow_pickle=True)

X_train_final = data['X_train']
X_test_final = data['X_test']
y_train = data['y_train']
y_test = data['y_test']

X_ucl = data['X_ucl']
y_ucl_true = data['y_ucl']

feature_names = np.load('feature_names.npy', allow_pickle=True)

meta_test = pd.read_csv('meta_test.csv')
meta_ucl = pd.read_csv('meta_ucl.csv')
club_names = pd.read_csv('club_names.csv')
id_to_name = dict(zip(club_names['club_id'], club_names['name']))

print(X_train_final.shape, X_test_final.shape, X_ucl.shape, len(feature_names))

(61873, 319) (15469, 319) (189, 319) 319


## Encode the target

home_win / draw / away_win -> 0 / 1 / 2

In [17]:
label_encoder = LabelEncoder()
y_train_enc = label_encoder.fit_transform(y_train)
y_test_enc = label_encoder.transform(y_test)

print(dict(zip(label_encoder.classes_, range(len(label_encoder.classes_)))))

{'away_win': 0, 'draw': 1, 'home_win': 2}


## Class weights

Draws are rare. This tells the model to care about them more. Set False to turn off.

In [18]:
USE_WEIGHTS = True
sample_weights = (compute_sample_weight(class_weight='balanced', y=y_train_enc)
                  if USE_WEIGHTS else None)

## Train

In [19]:
xgb_model = XGBClassifier(
    n_estimators=400,
    max_depth=5,
    learning_rate=0.03,
    objective='multi:softmax',
    num_class=3,
    eval_metric='mlogloss',
    random_state=42
)

xgb_model.fit(X_train_final, y_train_enc, sample_weight=sample_weights)

y_pred = label_encoder.inverse_transform(xgb_model.predict(X_test_final))
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

    away_win       0.57      0.50      0.53      5225
        draw       0.28      0.52      0.36      3379
    home_win       0.67      0.46      0.55      6865

    accuracy                           0.49     15469
   macro avg       0.51      0.49      0.48     15469
weighted avg       0.55      0.49      0.50     15469

[[2597 1854  774]
 [ 857 1742  780]
 [1091 2587 3187]]


## Grid search

Commented out, takes ages. Best params noted at the bottom.

In [20]:
'''
param_grid = {
    'n_estimators': [400, 500],
    'max_depth': [5, 8]
}

from sklearn.model_selection import TimeSeriesSplit
tscv = TimeSeriesSplit(n_splits=5)

xgb_model = XGBClassifier(
    learning_rate=0.03,
    objective='multi:softmax',
    num_class=3,
    eval_metric='mlogloss',
    random_state=42
)

grid_search = GridSearchCV(xgb_model, param_grid=param_grid, cv=tscv, scoring='f1_macro', verbose=2, refit=True)

grid_search.fit(X_train_final, y_train_enc, sample_weight=sample_weights)

y_pred_enc = grid_search.predict(X_test_final)
y_pred = label_encoder.inverse_transform(y_pred_enc)

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
'''

# best params: {'max_depth': 5, 'n_estimators': 400}

"\nparam_grid = {\n    'n_estimators': [400, 500],\n    'max_depth': [5, 8]\n}\n\nfrom sklearn.model_selection import TimeSeriesSplit\ntscv = TimeSeriesSplit(n_splits=5)\n\nxgb_model = XGBClassifier(\n    learning_rate=0.03,\n    objective='multi:softmax',\n    num_class=3,\n    eval_metric='mlogloss',\n    random_state=42\n)\n\ngrid_search = GridSearchCV(xgb_model, param_grid=param_grid, cv=tscv, scoring='f1_macro', verbose=2, refit=True)\n\ngrid_search.fit(X_train_final, y_train_enc, sample_weight=sample_weights)\n\ny_pred_enc = grid_search.predict(X_test_final)\ny_pred = label_encoder.inverse_transform(y_pred_enc)\n\nprint(classification_report(y_test, y_pred))\nprint(confusion_matrix(y_test, y_pred))\n"

## Feature importance

Also commented out, slow.

In [21]:
'''
perm_result = permutation_importance(
    xgb_model, X_test_final, y_test_enc,
    scoring='f1_macro', n_repeats=10, random_state=42
)

perm_df = pd.DataFrame({
    'feature': feature_names,
    'importance_mean': perm_result.importances_mean,
    'importance_std': perm_result.importances_std
}).sort_values('importance_mean', ascending=False)

print(perm_df.head(20))
'''

"\nperm_result = permutation_importance(\n    xgb_model, X_test_final, y_test_enc,\n    scoring='f1_macro', n_repeats=10, random_state=42\n)\n\nperm_df = pd.DataFrame({\n    'feature': feature_names,\n    'importance_mean': perm_result.importances_mean,\n    'importance_std': perm_result.importances_std\n}).sort_values('importance_mean', ascending=False)\n\nprint(perm_df.head(20))\n"

## Test on the 2025/26 Champions League

189 games the model has never seen. Compare the accuracy to the baseline, not to 100%.

In [22]:
y_ucl_pred = label_encoder.inverse_transform(xgb_model.predict(X_ucl))

baseline = pd.Series(y_ucl_true).value_counts(normalize=True).max()
print(f"accuracy: {(y_ucl_pred == y_ucl_true).mean():.2%}   (majority baseline {baseline:.2%})")
print(classification_report(y_ucl_true, y_ucl_pred))

accuracy: 52.91%   (majority baseline 49.74%)
              precision    recall  f1-score   support

    away_win       0.49      0.62      0.55        61
        draw       0.12      0.12      0.12        34
    home_win       0.73      0.62      0.67        94

    accuracy                           0.53       189
   macro avg       0.45      0.45      0.45       189
weighted avg       0.55      0.53      0.53       189



## Game by game

In [23]:
probs = xgb_model.predict_proba(X_ucl)
cls = list(label_encoder.classes_)

results = meta_ucl.copy()
results['home'] = results['home_club_id'].map(id_to_name)
results['away'] = results['away_club_id'].map(id_to_name)
results['actual'] = y_ucl_true
results['pred'] = y_ucl_pred
results['p_home'] = probs[:, cls.index('home_win')]
results['p_draw'] = probs[:, cls.index('draw')]
results['p_away'] = probs[:, cls.index('away_win')]

results['pred_winner'] = np.where(
    results['pred'] == 'home_win', results['home'],
    np.where(results['pred'] == 'away_win', results['away'], 'Draw'))

results[['date', 'round', 'home', 'away', 'actual', 'pred', 'pred_winner']].head(15)

,date,round,home,away,actual,pred,pred_winner
0,2025-09-16,Group Stage,PSV Eindhoven,Union Saint-Gilloise,away_win,home_win,PSV Eindhoven
1,2025-09-16,Group Stage,Real Madrid,Olympique Marseille,home_win,home_win,Real Madrid
2,2025-09-16,Group Stage,SL Benfica,Qarabağ FK,away_win,home_win,SL Benfica
3,2025-09-16,Group Stage,Athletic Bilbao,Arsenal FC,away_win,away_win,Arsenal FC
4,2025-09-16,Group Stage,Juventus FC,Borussia Dortmund,draw,home_win,Juventus FC
5,2025-09-16,Group Stage,Tottenham Hotspur,Villarreal CF,home_win,home_win,Tottenham Hotspur
6,2025-09-17,Group Stage,Bayern Munich,Chelsea FC,home_win,draw,Draw
7,2025-09-17,Group Stage,Liverpool FC,Atlético de Madrid,home_win,home_win,Liverpool FC
8,2025-09-17,Group Stage,Ajax Amsterdam,Inter Milan,away_win,away_win,Inter Milan
9,2025-09-17,Group Stage,Olympiakos Syndesmos Filathlon Peiraios,Pafos FC,draw,home_win,Olympiakos Syndesmos Filathlon Peiraios


## Accuracy by stage

Group stage vs knockouts.

In [24]:
# which rounds does it actually get right?
results['correct'] = results['pred'] == results['actual']
by_stage = (results.assign(stage=np.where(results['is_knockout'] == 1, 'knockout', 'group'))
            .groupby(['stage', 'leg'])['correct']
            .agg(['mean', 'size'])
            .rename(columns={'mean': 'accuracy', 'size': 'matches'}))
print(by_stage.round(3))

              accuracy  matches
stage    leg                   
group    0       0.528      144
knockout 0       0.000        1
         1       0.409       22
         2       0.682       22


## Simulating two-legged ties

Who goes through, not who wins a single game.

In [25]:
rng = np.random.default_rng(42)
N_SIMS = 20000
MARGIN = {'home_win': 1, 'draw': 0, 'away_win': -1}

def simulate_tie(legs):
    """legs: rows of `results` for one tie. Returns P(advance) per club_id."""
    ref = legs.iloc[0]['home_club_id']          # pick one team and track them
    totals = np.zeros(N_SIMS)

    for _, leg in legs.iterrows():
        p = leg[['p_away', 'p_draw', 'p_home']].values.astype(float)
        p = p / p.sum()
        drawn = rng.choice(['away_win', 'draw', 'home_win'], size=N_SIMS, p=p)
        margin = np.vectorize(MARGIN.get)(drawn)
        # flip the sign when they're playing away
        totals += margin if leg['home_club_id'] == ref else -margin

    # if it's level, coin flip (pretend that's extra time / pens)
    coin = rng.integers(0, 2, size=N_SIMS)
    ref_adv = (totals > 0) | ((totals == 0) & (coin == 1))
    other = legs.iloc[0]['away_club_id'] if legs.iloc[0]['home_club_id'] == ref else legs.iloc[0]['home_club_id']
    return {ref: ref_adv.mean(), other: 1 - ref_adv.mean()}


two_legged = results[results['leg'] > 0]
rows = []

for tie_key, legs in two_legged.groupby('tie_key'):
    if len(legs) != 2:
        continue
    legs = legs.sort_values('date')
    p_adv = simulate_tie(legs)

    first = legs.iloc[0]
    a, b = first['home_club_id'], first['away_club_id']
    pred_adv = max(p_adv, key=p_adv.get)

    rows.append({
        'tie': f"{id_to_name.get(a, a)} vs {id_to_name.get(b, b)}",
        'round': first['round'],
        'pred_advance': id_to_name.get(pred_adv, pred_adv),
        'p': round(max(p_adv.values()), 3),
    })

tie_df = pd.DataFrame(rows)
print(f"{len(tie_df)} two-legged ties")
tie_df

22 two-legged ties


,tie,round,pred_advance,p
0,Club Brugge KV vs Atlético de Madrid,intermediate stage 1st leg,Atlético de Madrid,0.727
1,Galatasaray vs Juventus FC,intermediate stage 1st leg,Juventus FC,0.543
2,Olympiakos Syndesmos Filathlon Peiraios vs Bay...,intermediate stage 1st leg,Bayer 04 Leverkusen,0.654
3,AS Monaco vs Paris Saint-Germain,intermediate stage 1st leg,Paris Saint-Germain,0.835
4,Borussia Dortmund vs Atalanta BC,intermediate stage 1st leg,Borussia Dortmund,0.585
5,SL Benfica vs Real Madrid,intermediate stage 1st leg,Real Madrid,0.672
6,Fotballklubben Bodø/Glimt vs Inter Milan,intermediate stage 1st leg,Inter Milan,0.821
7,Qarabağ FK vs Newcastle United,intermediate stage 1st leg,Newcastle United,0.877
8,Bayer 04 Leverkusen vs Arsenal FC,Last 16 1st Leg,Arsenal FC,0.684
9,Newcastle United vs FC Barcelona,Last 16 1st Leg,FC Barcelona,0.701


## The Final

PSG vs Arsenal. Note 583 is the home club and 11 is away — read the names, not the label.

In [26]:
final = results.loc[results['date'].idxmax()]

print(f"{final['home']}  vs  {final['away']}   ({final['date']}, {final['round']})")
print(f"  home {final['p_home']:.3f} | draw {final['p_draw']:.3f} | away {final['p_away']:.3f}")

champion = final['home'] if final['p_home'] > final['p_away'] else final['away']
gap = abs(final['p_home'] - final['p_away'])
print(f"\npredicted champion: {champion}   (margin over the other side: {gap:.3f})")
if gap < 0.05:
    print("  -> effectively a coin flip; not a meaningful call")

Paris Saint-Germain  vs  Arsenal FC   (2026-05-30, Final)
  home 0.498 | draw 0.027 | away 0.475

predicted champion: Paris Saint-Germain   (margin over the other side: 0.022)
  -> effectively a coin flip; not a meaningful call


## Find the club ids

In [27]:
# look up the ids
print(club_names[club_names['name'].str.contains('Barcelona|Sevilla', case=False, na=False)])

     club_id                    name
260      368              Sevilla FC
461      131            FC Barcelona
501      714  RCD Espanyol Barcelona


## Does it get easy games right?

Barcelona vs Sevilla should be obvious. Checking the model isn't just guessing 50/50 on everything.

In [ ]:
BAR, SEV = 131, 368   

pair = (meta_test['home_club_id'].isin([BAR, SEV]) &
        meta_test['away_club_id'].isin([BAR, SEV]))
idx = np.where(pair)[0]

p = xgb_model.predict_proba(X_test_final[idx])
cls = list(label_encoder.classes_)

out = meta_test.loc[pair, ['date', 'competition_id', 'home_club_id', 'away_club_id']].copy()
out['home'] = out['home_club_id'].map(id_to_name)
out['away'] = out['away_club_id'].map(id_to_name)
out['actual'] = y_test[idx]
out['p_home'] = p[:, cls.index('home_win')].round(3)
out['p_draw'] = p[:, cls.index('draw')].round(3)
out['p_away'] = p[:, cls.index('away_win')].round(3)

out[['date','home','away','actual','p_home','p_draw','p_away']]

,date,home,away,actual,p_home,p_draw,p_away
2408,2024-10-20,FC Barcelona,Sevilla FC,home_win,0.676,0.249,0.075
5125,2025-02-09,Sevilla FC,FC Barcelona,away_win,0.108,0.321,0.571
10060,2025-10-05,Sevilla FC,FC Barcelona,home_win,0.125,0.335,0.540
14010,2026-03-15,FC Barcelona,Sevilla FC,home_win,0.702,0.244,0.054


Good — confident when it should be, unsure when it should be.

---

## Honest summary

**Works:** small edge over guessing, sensible probabilities on obvious games.

**Doesn't:** draws are bad (F1 ~0.15). Two-legged tie features did nothing. Can't predict penalty shootouts.

**Could try next:** probability calibration, UCL-only Elo, predicting scorelines instead of win/draw/loss.